# 02 - Silver: CIIU (Catálogo de Actividades Económicas)

Trabajo Práctico - Pipeline ETL end-to-end (Bronze → Silver / DQL → Gold)

| | Tabla |
|---|---|
| Entrada | `datos_empresas.bronze.de_semiestructurado_actividadeseconomicas_api` |
| Salida Silver | `datos_empresas.silver.de_actividadeseconomicas` |
| Salida DQL | `datos_empresas.dql.de_actividadeseconomicas_dql` |
| Llave (grano) | `codigo_ciiu` |

Se procesa **antes que RUES** porque la regla de correspondencia de RUES valida cada empresa contra este catálogo ya depurado.

## Transformaciones

| Bronze | Silver | Transformación |
|---|---|---|
| `code` | `codigo_ciiu` | `trim` |
| `description` | `descripcion_ciiu` | `trim` |
| `id` | `id_ciiu` | texto → `INT` |
| `status` | `estado_ciiu` | texto → `INT` |
| `version` | `version_ciiu` | texto → `INT` |

## Reglas de calidad (rechazo hacia DQL)

| # | Regla (`_motivo_rechazo`) | Condición | Por qué es pertinente |
|---|---|---|---|
| 1 | `codigo_ciiu_invalido` | Código vacío o que no tiene exactamente 4 dígitos | El código CIIU Rev. 4 A.C. es una clase de 4 dígitos. Es la llave con la que RUES se enlaza al catálogo; un código mal formado nunca coincidiría y contaminaría la regla de correspondencia de RUES. |
| 2 | `descripcion_vacia` | Descripción vacía | El catálogo existe para traducir el código a una actividad legible en Gold. Un código sin descripción no aporta nada al análisis. |
| 3 | `duplicado_version_anterior` | Hay otra versión más reciente del mismo código | Unicidad: una sola descripción por código. Un código repetido duplicaría empresas al hacer `JOIN` con RUES en Gold. Se conserva la versión más reciente (`version` y `_ingested_at` mayores). |

In [ ]:
%run ./00_Utilidades_Pipeline

---
## Paso 1: Lectura de Bronze y tipado

In [ ]:
%python
df_bronze = preparar_bronze(spark.table(BRONZE_CIIU))

df_tipado = df_bronze.select(
    "_registro_bronze", "_hash_bronze", "_id_registro_bronze",
    F.col("_ingested_at").alias("_bronze_ingested_at"),
    texto_limpio("code").alias("codigo_ciiu"),
    texto_limpio("description").alias("descripcion_ciiu"),
    F.expr("try_cast(id AS INT)").alias("id_ciiu"),
    F.expr("try_cast(status AS INT)").alias("estado_ciiu"),
    F.expr("try_cast(version AS INT)").alias("version_ciiu"),
)

df_versionado = marcar_version(
    df_tipado, ["codigo_ciiu"],
    [F.col("version_ciiu").desc_nulls_last(), F.col("_bronze_ingested_at").desc()],
)

---
## Paso 1.5: Diagnóstico previo — evidencia detrás de cada regla

Resultado con el Bronze del 21-oct-2026 (499 filas): 0 en todas las reglas. El catálogo llega limpio; las reglas quedan como salvaguarda para cargas futuras.

In [ ]:
%sql
-- Evidencia: filas de Bronze que incumplirían cada regla (mismas condiciones que el Paso 2)
WITH c AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY TRIM(code)
                            ORDER BY TRY_CAST(version AS INT) DESC NULLS LAST, _ingested_at DESC) AS version_fila
  FROM datos_empresas.bronze.de_semiestructurado_actividadeseconomicas_api
)
SELECT
  COUNT(*) AS filas_bronze,
  SUM(CASE WHEN TRIM(COALESCE(code, '')) = '' OR NOT TRIM(code) RLIKE '^[0-9]{4}$' THEN 1 ELSE 0 END) AS r1_codigo_ciiu_invalido,
  SUM(CASE WHEN TRIM(COALESCE(description, '')) = '' THEN 1 ELSE 0 END) AS r2_descripcion_vacia,
  SUM(CASE WHEN version_fila > 1 THEN 1 ELSE 0 END) AS r3_duplicado_version_anterior
FROM c;

---
## Paso 2: Reglas de calidad y separación Silver / DQL

In [ ]:
%python
REGLAS = [
    ("codigo_ciiu_invalido", es_vacio("codigo_ciiu") | ~F.coalesce(F.col("codigo_ciiu").rlike("^[0-9]{4}$"), F.lit(False))),
    ("descripcion_vacia", es_vacio("descripcion_ciiu")),
    ("duplicado_version_anterior", (F.col("_version") > 1) & ~es_vacio("codigo_ciiu")),
]

df_aceptados, df_rechazados = separar_aceptados_rechazados(df_versionado, REGLAS)

COLUMNAS_SILVER = ["codigo_ciiu", "descripcion_ciiu", "id_ciiu", "estado_ciiu", "version_ciiu"]

df_silver = construir_silver(df_aceptados, COLUMNAS_SILVER)
df_dql = construir_dql(df_rechazados)

---
## Paso 3: Escritura idempotente (MERGE) en Silver y DQL

In [ ]:
%python
m_silver = escribir_merge(
    df_silver, SILVER_CIIU, ["codigo_ciiu"], capa="silver",
    comentario="Silver - Catalogo CIIU Rev. 4 depurado: un registro por codigo de actividad economica.",
)
m_dql = escribir_merge(
    df_dql, DQL_CIIU, ["_id_registro_bronze"], capa="dql", columna_fecha="_fecha_rechazo",
    comentario="DQL - Registros del catalogo CIIU rechazados, con la fila original de Bronze y el motivo.",
)

conteos = conciliar_fuente("CIIU")

---
## Paso 4: Validación

In [ ]:
%sql
SELECT _motivo_rechazo, COUNT(*) AS registros
FROM datos_empresas.dql.de_actividadeseconomicas_dql
GROUP BY _motivo_rechazo
ORDER BY registros DESC;

In [ ]:
%sql
SELECT * FROM datos_empresas.silver.de_actividadeseconomicas ORDER BY codigo_ciiu LIMIT 10;

In [ ]:
%python
finalizar({"notebook": "02_Silver_CIIU", "conteos": conteos, "silver": m_silver, "dql": m_dql})